In [9]:
import pandas as pd
import numpy as np
from astropy.io import fits

In [10]:
# List of your JADES catalog files
file_paths = [
    "/Users/aryanahaghjoo/Documents/GitHub/super_resolution/data/JADES/catalogue/jades_dr3_prism_public_gn_v1.1.fits",
    "/Users/aryanahaghjoo/Documents/GitHub/super_resolution/data/JADES/catalogue/jades_dr3_prism_public_gs_v1.1.fits"
]

# Columns you always want
base_cols = ["RA_TARG", "Dec_TARG", "Field", "z_Spec", "z_PRISM"]

df_list = []
for path in file_paths:
    with fits.open(path) as hdul:
        data    = hdul[1].data
        df_temp = pd.DataFrame(np.array(data))

        # Decode Field if byte‐string
        if "Field" in df_temp.columns:
            df_temp["Field"] = df_temp["Field"].apply(
                lambda x: x.decode("utf-8") if isinstance(x, bytes) else x
            )

        # Find any NIRCam‐ID columns
        nircam_cols = [c for c in df_temp.columns if "nircam" in c.lower()]

        # Build the list of columns to keep
        cols_to_keep = base_cols + nircam_cols
        existing_cols = [c for c in cols_to_keep if c in df_temp.columns]

        # Subset
        df_sub = df_temp[existing_cols].copy()

        # If your NIRCam IDs are byte‐strings, decode them too
        for c in nircam_cols:
            if df_sub[c].dtype == object:
                df_sub[c] = df_sub[c].apply(
                    lambda x: x.decode("utf-8") if isinstance(x, bytes) else x
                )

        df_list.append(df_sub)

# Combine all fields
df_redshifts = pd.concat(df_list, ignore_index=True)

# Apply your existing filter
mask = ~((df_redshifts["z_Spec"] == -1) & (df_redshifts["z_PRISM"].isna()))
df_redshifts = df_redshifts[mask].reset_index(drop=True)

In [11]:
df_redshifts

,RA_TARG,Dec_TARG,Field,z_Spec,z_PRISM,NIRCam_ID,RA_NIRCam,Dec_NIRCam
0,189.137065,62.213273,GN,3.909444,3.917421,1000033,189.137063,62.213274
1,189.131012,62.213989,GN,2.442181,2.441134,1000058,189.131020,62.213989
2,189.129198,62.215141,GN,3.909191,3.914008,1000095,189.129203,62.215139
3,189.146379,62.215508,GN,4.064162,4.073715,1000110,189.146379,62.215508
4,189.123825,62.215497,GN,5.788500,5.795857,1000113,189.123835,62.215495
...,...,...,...,...,...,...,...,...
2725,53.157059,-27.772718,GS,5.972446,5.977236,131701,53.157060,-27.772718
2726,53.186276,-27.779041,GS,7.273864,7.278073,127088,53.186276,-27.779040
2727,53.136002,-27.798489,GS,5.777481,5.778242,202809,53.136000,-27.798489
2728,53.131975,-27.779204,GS,5.545532,5.553566,208359,53.131977,-27.779192


In [12]:
#either z_spec or z_prism is >=10

# Assuming your original DataFrame is named df
df_highz_or = df_redshifts[(df_redshifts['z_Spec'] >= 10) | (df_redshifts['z_PRISM'] >= 10)]

In [13]:
df_highz_or

,RA_TARG,Dec_TARG,Field,z_Spec,z_PRISM,NIRCam_ID,RA_NIRCam,Dec_NIRCam
186,189.106054,62.242049,GN,10.605965,10.609332,1005591,189.106047,62.242041
213,189.106043,62.242045,GN,10.604423,10.608815,1005591,189.106047,62.242041
1034,189.286085,62.169879,GN,11.167000,11.167000,1119489,189.286092,62.169881
1177,53.166338,-27.821555,GS,12.630000,12.630000,96216,53.166345,-27.821558
1713,53.149880,-27.776500,GS,13.200000,13.200000,128771,53.149882,-27.776502
2659,53.158840,-27.773492,GS,10.409450,10.409450,131067,53.158835,-27.773495
2660,53.164768,-27.774626,GS,11.667940,11.667940,130158,53.164763,-27.774622
2688,53.165939,-27.834242,GS,10.225570,10.225570,86029,53.165935,-27.834237
2689,53.165939,-27.834242,GS,10.562810,10.562810,86029,53.165935,-27.834237
2704,53.166346,-27.821557,GS,12.473683,12.473683,96216,53.166345,-27.821558


In [14]:
#both z_spec and z_prism is >=10
df_highz_and = df_redshifts[(df_redshifts['z_Spec'] >= 10) & (df_redshifts['z_PRISM'] >= 10)]

In [15]:
df_highz_and

,RA_TARG,Dec_TARG,Field,z_Spec,z_PRISM,NIRCam_ID,RA_NIRCam,Dec_NIRCam
186,189.106054,62.242049,GN,10.605965,10.609332,1005591,189.106047,62.242041
213,189.106043,62.242045,GN,10.604423,10.608815,1005591,189.106047,62.242041
1034,189.286085,62.169879,GN,11.167000,11.167000,1119489,189.286092,62.169881
1177,53.166338,-27.821555,GS,12.630000,12.630000,96216,53.166345,-27.821558
1713,53.149880,-27.776500,GS,13.200000,13.200000,128771,53.149882,-27.776502
2659,53.158840,-27.773492,GS,10.409450,10.409450,131067,53.158835,-27.773495
2660,53.164768,-27.774626,GS,11.667940,11.667940,130158,53.164763,-27.774622
2688,53.165939,-27.834242,GS,10.225570,10.225570,86029,53.165935,-27.834237
2689,53.165939,-27.834242,GS,10.562810,10.562810,86029,53.165935,-27.834237
2704,53.166346,-27.821557,GS,12.473683,12.473683,96216,53.166345,-27.821558


In [18]:
df_highz_and.to_csv('high_z_objects.csv')

Both of the conditions gave the same number of galaxies.